In [ ]:
ltm_base = "ltm-p-0.1-q-0.7_1"
icc_base = "icc-p-0.3-q-0.7_1"
dcc_base = "dcc-p-0.1-q-0.7_1"

import networkx as nx
import pandas as pd
import scienceplots
import matplotlib.pyplot as plt
plt.style.use(["science", "no-latex"])


datasets = ["Enron-MHDA-1000", "Twitch-MHDA-1000", "Physics-MHDA-1000", "P2P-MHDA-1000", "Twitter-MHDA-1000", "Airports-MHDA-1000"]
heuristics = ["CELF", "Degree", "Deg-div-mnd"]

ltm = {}
dcc = {}
icc = {}
for dataset in datasets:
    ltm[dataset.split("-")[0]] = [pd.read_pickle(f"{ltm_base}/Compiled-LTM-{dataset}-{heuristic}.pickle") for heuristic in heuristics]
    icc[dataset.split("-")[0]] = [pd.read_pickle(f"{icc_base}/Compiled-ICC-{dataset}-{heuristic}.pickle") for heuristic in heuristics]
    dcc[dataset.split("-")[0]] = [pd.read_pickle(f"{dcc_base}/Compiled-DCC-{dataset}-{heuristic}.pickle") for heuristic in heuristics]

icc

In [ ]:
def draw_dataset_onto_axis(ax, dataset, title="", tickers=["o", "v", "D"]):
    n_s = dataset[0]["n"][::-1].to_numpy()
    n_s = n_s.astype(int)
    methods = [s['heuristic'].unique()[0] for s in dataset]
    #print(n_s)
#    if baseline != "":
#        baseline = (baseline, dataset[[i for i, v in enumerate(dataset) if v["heuristic"].unique()[0] == baseline][0]]["end_coverage"].to_numpy()[::-1])
    #print(methods)
    for i, v in enumerate(dataset):
        ax.plot(n_s, v['end_coverage'].to_numpy()[::-1], label=methods[i], marker=tickers[i % len(tickers)], markersize=4)
    ax.legend()
    ax.set_xlabel("Starting Seed Size")
    #ax.set_ylabel(f"Delta to baseline {baseline[0]}")
    ax.set_title(title)
    ax.set_ylabel("Final Influence Spread")

In [ ]:
def draw_all_6(data, networks, label):
    fig, axs = plt.subplots(2, 3)
    fig.set_dpi(400)
    fig.set_size_inches(12, 9)
    indicies = [(x, y) for x in range(2) for y in range(3)]
    for i, net in enumerate(networks):
        draw_dataset_onto_axis(axs[indicies[i]], data[net.split("-")[0]], title=f"{net} {label}")
    plt.savefig(f"{label.strip()}-performance.png")
for label, model in [("ICC p=0.3 n=100", icc), ("DCC p=0.1 q=0.7 n=100", dcc), ("LTM theta=0.95 n=500", ltm)]:
    draw_all_6(model, datasets, label)

In [ ]:
x = pd.read_pickle("icc-p-0.3-q-0.7_1/All-ICC-Compiled-Tables.pickle")

In [ ]:
monte = pd.read_pickle("montecarlo-icc/All-ICC-Compiled-Tables.pickle")
monte

In [ ]:
def draw_dataset_onto_axis_s(ax, dataset, title="", tickers=["o", "v", "D"]):
    n_s = dataset[0]["n"][::-1].to_numpy()
    n_s = n_s.astype(int)
    methods = [s['heuristic'].unique()[0] for s in dataset]
    for i, v in enumerate(dataset):
        line = v['end_coverage'].to_numpy()[::-1]
        above = line + v["end_coverage_std"].to_numpy()[::-1]
        below = line - v["end_coverage_std"].to_numpy()[::-1]
        
        #ax.plot(n_s, above, label=methods[i], marker=tickers[i % len(tickers)], markersize=4)
        ax.plot(n_s, line, label=methods[i], marker=tickers[i % len(tickers)], markersize=4)
        ax.fill_between(n_s, below, above, alpha=0.3)
        #ax.plot(n_s, below, label=methods[i], marker=tickers[i % len(tickers)], markersize=4)
    ax.legend()
    ax.set_xlabel("Starting Seed Size")
    #ax.set_ylabel(f"Delta to baseline {baseline[0]}")
    ax.set_title(title)
    ax.set_ylabel("Final Influence Spread")
celf = monte[monte["heuristic"] == "CELF"]
mon = monte[monte["heuristic"] != "CELF"]
fig, ax = plt.subplots()
fig.set_dpi(400)
fig.set_size_inches(5, 5)
draw_dataset_onto_axis_s(ax, [mon, celf])
ax.set_title("Comparison of CELF with and without Monte Carlo Method")
plt.savefig("celf-mc-vs-non-mc.png")

In [ ]:
import os
import pickle
import re
from collections import defaultdict
import numpy as np
from scipy.stats import mannwhitneyu
import pandas as pd

out_dir = "montecarlo-icc"  # <-- change this

# Pattern: ICC-CELF[-MonteCarlo]-{dataset}-{network_sample}-{simulations}-{seed_size}-{run}.pickle
pattern = re.compile(
    r"^ICC-(CELF(?:-MonteCarlo)?)-(\w+)-(\w+)-(\d+)-(\d+)-(\d+)\.pickle$"
)

results = defaultdict(list)

for fname in os.listdir(out_dir):
    match = pattern.match(fname)
    if not match:
        continue

    model, dataset, network_sample, simulations, seed_size, run = match.groups()
    key = (model, dataset, network_sample, simulations, seed_size)

    fpath = os.path.join(out_dir, fname)
    try:
        with open(fpath, "rb") as f:
            df = pickle.load(f)
        end_coverage = float(df["end_coverage"].iloc[-1])
        results[key].append(end_coverage)
    except Exception as e:
        print(f"Error reading {fname}: {e}")

# --- Mann-Whitney U tests: CELF vs CELF-MonteCarlo for each (dataset, network_sample, simulations, seed_size) ---

# Get all unique non-model keys
unique_configs = set()
for (model, dataset, network_sample, simulations, seed_size) in results.keys():
    unique_configs.add((dataset, network_sample, simulations, seed_size))

mw_results = []

for (dataset, network_sample, simulations, seed_size) in sorted(unique_configs):
    key_celf = ("CELF",             dataset, network_sample, simulations, seed_size)
    key_mc   = ("CELF-MonteCarlo",  dataset, network_sample, simulations, seed_size)

    if key_celf not in results:
        print(f"Missing CELF for {dataset}, {network_sample}, sims={simulations}, seed={seed_size}")
        continue
    if key_mc not in results:
        print(f"Missing CELF-MonteCarlo for {dataset}, {network_sample}, sims={simulations}, seed={seed_size}")
        continue

    arr_celf = np.array(results[key_celf])
    arr_mc   = np.array(results[key_mc])

    stat, p_value = mannwhitneyu(arr_celf, arr_mc, alternative="two-sided")
    same = p_value > 0.05

    mw_results.append({
        "dataset":          dataset,
        "network_sample":   network_sample,
        "simulations":      simulations,
        "seed_size":        int(seed_size),
        "n_celf":           len(arr_celf),
        "n_mc":             len(arr_mc),
        "mean_celf":        arr_celf.mean(),
        "mean_mc":          arr_mc.mean(),
        "U_statistic":      stat,
        "p_value":          p_value,
        "same_distribution": same
    })

    print(
        f"{dataset} | {network_sample} | sims={simulations} | seed={seed_size:>4} | "
        f"mean_celf={arr_celf.mean():.4f}, mean_mc={arr_mc.mean():.4f} | "
        f"U={stat:.1f}, p={p_value:.4f} | {'SAME ✓' if same else 'DIFFERENT ✗'}"
    )

# --- Summary DataFrame ---
summary_df = (
    pd.DataFrame(mw_results)
    .sort_values(["dataset", "network_sample", "simulations", "seed_size"])
    .reset_index(drop=True)
)

print("\n", summary_df.to_string(index=False))

In [ ]:
from statsmodels.stats.power import TTestIndPower
import numpy as np

# First calculate your observed effect size (Cohen's d) from your existing data
def cohens_d(a, b):
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return abs(np.mean(a) - np.mean(b)) / pooled_std

# For each config, calculate required sample size
for (dataset, network_sample, simulations, seed_size) in sorted(unique_configs):
    key_celf = ("CELF",            dataset, network_sample, simulations, seed_size)
    key_mc   = ("CELF-MonteCarlo", dataset, network_sample, simulations, seed_size)

    if key_celf not in results or key_mc not in results:
        continue

    arr_celf = np.array(results[key_celf])
    arr_mc   = np.array(results[key_mc])

    d = cohens_d(arr_celf, arr_mc)

    analysis = TTestIndPower()
    required_n = analysis.solve_power(
        effect_size=d,
        alpha=0.05,       # significance level
        power=0.80,       # 80% chance of detecting a real difference
        alternative="two-sided"
    )

    print(
        f"seed={seed_size} | "
        f"observed Cohen's d={d:.3f} | "
        f"current n_celf={len(arr_celf)}, n_mc={len(arr_mc)} | "
        f"required n per group={np.ceil(required_n):.0f} | "
        f"{'sufficient' if len(arr_celf) >= required_n else 'insufficient'}"
    )

In [ ]:
import random
import networkx as nx

# Parameters
n = 10  # number of nodes
p = 0.3  # probability of edge between nodes

# Generate a random graph (Erdős-Rényi model)
G = nx.erdos_renyi_graph(n, p)

# Assign random values to each node
for node in G.nodes:
    G.nodes[node]['value'] = round(random.uniform(0, 1), 2)

# Print node values
print("Node values:")
for node, data in G.nodes(data=True):
    print(f"Node {node}: {data['value']}")
labels = {node: G.nodes[node]['value'] for node in G.nodes}
pos = nx.arf_layout(G)  # positions for all nodes
nx.draw(G, pos, with_labels=True, labels=labels, node_color='skyblue', node_size=800, font_size=12, font_weight='bold')
plt.show()

In [ ]:
import os
import pickle
import re
from collections import defaultdict
import numpy as np
from scipy.stats import mannwhitneyu
import pandas as pd
from itertools import combinations

out_dir = "ltm-p-0.1-q-0.7_1"  # <-- change this

# Pattern: {diffusion_model}-{algorithm}-{dataset}-{network_sample}-{simulations}-{seed_size}-{run}.pickle
pattern = re.compile(
    r"^(LTM)-(CELF|Degree|Deg-div-mnd)-(\w+)-(\w+)-(\d+)-(\d+)-(\d+)\.pickle$"
)

results = defaultdict(list)

for fname in os.listdir(out_dir):
    match = pattern.match(fname)
    if not match:
        continue

    diffusion, algorithm, dataset, network_sample, simulations, seed_size, run = match.groups()
    key = (diffusion, algorithm, dataset, network_sample, simulations, seed_size)

    fpath = os.path.join(out_dir, fname)
    try:
        with open(fpath, "rb") as f:
            df = pickle.load(f)
        end_coverage = float(df["end_coverage"].iloc[-1])
        results[key].append(end_coverage)
    except Exception as e:
        print(f"Error reading {fname}: {e}")

# --- Mann-Whitney U tests: pairwise between CELF, Degree, Deg-div-mnd ---
print("Begining TESTING")

algorithms = ["CELF", "Degree", "Deg-div-mnd"]
algo_pairs = list(combinations(algorithms, 2))  # (CELF vs Degree), (CELF vs Deg-div-mnd), (Degree vs Deg-div-mnd)

# Get all unique (diffusion, dataset, network_sample, simulations, seed_size) configs
unique_configs = set()
for (diffusion, algorithm, dataset, network_sample, simulations, seed_size) in results.keys():
    unique_configs.add((diffusion, dataset, network_sample, simulations, seed_size))

mw_results = []

print("PAIRWISE TEST")
for (diffusion, dataset, network_sample, simulations, seed_size) in sorted(unique_configs):
    for (algo_a, algo_b) in algo_pairs:
        key_a = (diffusion, algo_a, dataset, network_sample, simulations, seed_size)
        key_b = (diffusion, algo_b, dataset, network_sample, simulations, seed_size)

        if key_a not in results:
            print(f"Missing {algo_a} for {diffusion}, {dataset}, {network_sample}, sims={simulations}, seed={seed_size}")
            continue
        if key_b not in results:
            print(f"Missing {algo_b} for {diffusion}, {dataset}, {network_sample}, sims={simulations}, seed={seed_size}")
            continue

        arr_a = np.array(results[key_a])
        arr_b = np.array(results[key_b])

        stat, p_value = mannwhitneyu(arr_a, arr_b, alternative="two-sided")
        same = p_value > 0.05

        mw_results.append({
            "diffusion":        diffusion,
            "dataset":          dataset,
            "network_sample":   network_sample,
            "simulations":      simulations,
            "seed_size":        int(seed_size),
            "algo_a":           algo_a,
            "algo_b":           algo_b,
            "n_a":              len(arr_a),
            "n_b":              len(arr_b),
            "mean_a":           arr_a.mean(),
            "mean_b":           arr_b.mean(),
            "U_statistic":      stat,
            "p_value":          p_value,
            "same_distribution": same
        })

        print(
            f"{diffusion} | {dataset} | seed={seed_size:>4} | "
            f"{algo_a} vs {algo_b} | "
            f"mean_a={arr_a.mean():.4f}, mean_b={arr_b.mean():.4f} | "
            f"U={stat:.1f}, p={p_value} | {'SAME ✓' if same else 'DIFFERENT ✗'}"
        )

# --- Summary DataFrame ---
summary_df = (
    pd.DataFrame(mw_results)
    .sort_values(["diffusion", "dataset", "seed_size", "algo_a", "algo_b"])
    .reset_index(drop=True)
)

print("\n", summary_df.to_string(index=False))

# --- Pivot table for easy reading ---
pivot = summary_df.pivot_table(
    index=["dataset", "seed_size"],
    columns=["algo_a", "algo_b"],
    values="p_value"
).round(4)
max_p = summary_df["p_value"].max()
min_p = summary_df["p_value"].min()
print(max_p, min_p)
#print("\nP-value pivot table:")
#print(pivot.to_string())

In [ ]:
algo_a = "CELF"
algo_b = "Deg-div-mnd"

filtered = summary_df[
    ((summary_df["algo_a"] == algo_a) & (summary_df["algo_b"] == algo_b)) |
    ((summary_df["algo_a"] == algo_b) & (summary_df["algo_b"] == algo_a))
]

max_p = filtered["p_value"].max()
min_p = filtered["p_value"].min()
print(max_p, min_p)

In [ ]:
def cohens_d(a, b):
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return abs(np.mean(a) - np.mean(b)) / pooled_std

algo_a = "CELF"
algo_b = "Deg-div-mnd"

min_d = float("inf")
min_d_config = None

for (diffusion, dataset, network_sample, simulations, seed_size) in sorted(unique_configs):
    key_a = (diffusion, algo_a, dataset, network_sample, simulations, seed_size)
    key_b = (diffusion, algo_b, dataset, network_sample, simulations, seed_size)

    if key_a not in results or key_b not in results:
        continue

    arr_a = np.array(results[key_a])
    arr_b = np.array(results[key_b])

    d = cohens_d(arr_a, arr_b)

    if d < min_d:
        min_d = d
        min_d_config = {
            "diffusion":      diffusion,
            "dataset":        dataset,
            "network_sample": network_sample,
            "simulations":    simulations,
            "seed_size":      seed_size,
            "mean_a":         arr_a.mean(),
            "mean_b":         arr_b.mean(),
            "cohens_d":       d
        }

print(f"Minimum Cohen's d between {algo_a} and {algo_b}: {min_d:.4f}")
print("\nConfig:")
for k, v in min_d_config.items():
    print(f"  {k}: {v}")

from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

required_n = analysis.solve_power(
    effect_size=min_d,
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

print(f"Minimum Cohen's d:         {min_d:.4f}")
print(f"Required n per group:      {np.ceil(required_n):.0f}")
print(f"Total runs required:       {np.ceil(required_n) * 2:.0f}")

In [ ]:
import os
import pickle
import re
from collections import defaultdict
import numpy as np
from scipy.stats import mannwhitneyu
import pandas as pd
from itertools import combinations

out_dir = "dcc-p-0.1-q-0.7_1"  # <-- change this

# Pattern: {diffusion_model}-{algorithm}-{dataset}-{network_sample}-{simulations}-{seed_size}-{run}.pickle
pattern = re.compile(
    r"^(DCC)-(CELF|Degree|Deg-div-mnd)-(\w+)-(\w+)-(\d+)-(\d+)-(\d+)\.pickle$"
)

results = defaultdict(list)

for fname in os.listdir(out_dir):
    match = pattern.match(fname)
    if not match:
        continue

    diffusion, algorithm, dataset, network_sample, simulations, seed_size, run = match.groups()
    key = (diffusion, algorithm, dataset, network_sample, simulations, seed_size)

    fpath = os.path.join(out_dir, fname)
    try:
        with open(fpath, "rb") as f:
            df = pickle.load(f)
        end_coverage = float(df["end_coverage"].iloc[-1])
        results[key].append(end_coverage)
    except Exception as e:
        print(f"Error reading {fname}: {e}")

print("Begining TESTING")

algorithms = ["CELF", "Deg-div-mnd"]
algo_pairs = list(combinations(algorithms, 2))  # (CELF vs Degree), (CELF vs Deg-div-mnd), (Degree vs Deg-div-mnd)

unique_configs = set()
for (diffusion, algorithm, dataset, network_sample, simulations, seed_size) in results.keys():
    unique_configs.add((diffusion, dataset, network_sample, simulations, seed_size))

mw_results = []

print("PAIRWISE TEST")
for (diffusion, dataset, network_sample, simulations, seed_size) in sorted(unique_configs):
    for (algo_a, algo_b) in algo_pairs:
        key_a = (diffusion, algo_a, dataset, network_sample, simulations, seed_size)
        key_b = (diffusion, algo_b, dataset, network_sample, simulations, seed_size)

        if key_a not in results:
            print(f"Missing {algo_a} for {diffusion}, {dataset}, {network_sample}, sims={simulations}, seed={seed_size}")
            continue
        if key_b not in results:
            print(f"Missing {algo_b} for {diffusion}, {dataset}, {network_sample}, sims={simulations}, seed={seed_size}")
            continue

        arr_a = np.array(results[key_a])
        arr_b = np.array(results[key_b])

        stat, p_value = mannwhitneyu(arr_a, arr_b, alternative="two-sided")
        same = p_value > 0.05

        mw_results.append({
            "diffusion":        diffusion,
            "dataset":          dataset,
            "network_sample":   network_sample,
            "simulations":      simulations,
            "seed_size":        int(seed_size),
            "algo_a":           algo_a,
            "algo_b":           algo_b,
            "n_a":              len(arr_a),
            "n_b":              len(arr_b),
            "mean_a":           arr_a.mean(),
            "mean_b":           arr_b.mean(),
            "U_statistic":      stat,
            "p_value":          p_value,
            "same_distribution": same
        })

        print(
            f"{diffusion} | {dataset} | seed={seed_size:>4} | "
            f"{algo_a} vs {algo_b} | "
            f"mean_a={arr_a.mean():.4f}, mean_b={arr_b.mean():.4f} | "
            f"U={stat:.1f}, p={p_value} | {'SAME ✓' if same else 'DIFFERENT ✗'}"
        )

# --- Summary DataFrame ---
summary_df = (
    pd.DataFrame(mw_results)
    .sort_values(["diffusion", "dataset", "seed_size", "algo_a", "algo_b"])
    .reset_index(drop=True)
)

print("\n", summary_df.to_string(index=False))

# --- Pivot table for easy reading ---
pivot = summary_df.pivot_table(
    index=["dataset", "seed_size"],
    columns=["algo_a", "algo_b"],
    values="p_value"
).round(4)
max_p = summary_df["p_value"].max()
min_p = summary_df["p_value"].min()
print(max_p, min_p)
#print("\nP-value pivot table:")
#print(pivot.to_string())

In [ ]:
algo_a = "CELF"
algo_b = "Deg-div-mnd"

min_d = float("inf")
min_d_config = None

for (diffusion, dataset, network_sample, simulations, seed_size) in sorted(unique_configs):
    key_a = (diffusion, algo_a, dataset, network_sample, simulations, seed_size)
    key_b = (diffusion, algo_b, dataset, network_sample, simulations, seed_size)

    if key_a not in results or key_b not in results:
        continue

    arr_a = np.array(results[key_a])
    arr_b = np.array(results[key_b])

    d = cohens_d(arr_a, arr_b)

    if d < min_d:
        min_d = d
        min_d_config = {
            "diffusion":      diffusion,
            "dataset":        dataset,
            "network_sample": network_sample,
            "simulations":    simulations,
            "seed_size":      seed_size,
            "mean_a":         arr_a.mean(),
            "mean_b":         arr_b.mean(),
            "cohens_d":       d
        }

print(f"Minimum Cohen's d between {algo_a} and {algo_b}: {min_d:.4f}")
print("\nConfig:")
for k, v in min_d_config.items():
    print(f"  {k}: {v}")

from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

required_n = analysis.solve_power(
    effect_size=min_d,
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

print(f"Minimum Cohen's d:         {min_d:.4f}")
print(f"Required n per group:      {np.ceil(required_n):.0f}")
print(f"Total runs required:       {np.ceil(required_n) * 2:.0f}")